# 3장. 데이터의 첫인상 읽기

## 0. 제출 정보
- 이름: 유은송
- GitHub ID: song-03
- 작성일: 2026.09.08.
- 최종 제출 URL: https://github.com/song-03/llm-data-analysis-study/blob/main/chapter03/chapter03.ipynb

###실습 준비

In [1]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)


def find_project_root(start: Path) -> Path:
    """현재 위치에서 위로 올라가며 data/raw 폴더가 있는 프로젝트 루트를 찾습니다."""
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "data" / "raw").exists() and (candidate / "book").exists():
            return candidate
    raise FileNotFoundError("프로젝트 루트를 찾지 못했습니다. 노트북을 저장소 안에서 실행해 주세요.")


PROJECT_ROOT = find_project_root(Path.cwd())
DATA_DIR = PROJECT_ROOT / "data" / "raw"

print("프로젝트 루트:", PROJECT_ROOT)
print("데이터 폴더:", DATA_DIR)


FileNotFoundError: 프로젝트 루트를 찾지 못했습니다. 노트북을 저장소 안에서 실행해 주세요.

## 2. 데이터 파일이 있는지 확인하기

파일 경로 오류는 초보자가 가장 자주 만나는 오류입니다. 데이터를 불러오기 전에 필요한 CSV 파일이 실제로 있는지 먼저 확인합니다.


In [ ]:
expected_files = [
    "customers.csv",
    "products.csv",
    "orders.csv",
    "order_items.csv",
]

file_check = pd.DataFrame({
    "file": expected_files,
    "path": [str(DATA_DIR / filename) for filename in expected_files],
    "exists": [(DATA_DIR / filename).exists() for filename in expected_files],
})

file_check


`exists`가 모두 `True`이면 다음 단계로 진행할 수 있습니다.

하나라도 `False`라면 데이터가 아직 생성되지 않았을 수 있습니다. 그 경우 터미널에서 아래 명령을 실행해 샘플 데이터를 생성합니다.

```bash
python scripts/generate_sample_data.py
```


## 3. CSV 파일을 DataFrame으로 불러오기

이제 4개의 CSV 파일을 pandas `DataFrame`으로 불러옵니다. 각 변수 이름은 파일 이름과 비슷하게 맞춰 두면 이후 코드를 읽기 쉽습니다.


In [ ]:
customers = pd.read_csv(DATA_DIR / "customers.csv")
products = pd.read_csv(DATA_DIR / "products.csv")
orders = pd.read_csv(DATA_DIR / "orders.csv")
order_items = pd.read_csv(DATA_DIR / "order_items.csv")

print("customers:", type(customers))
print("products:", type(products))
print("orders:", type(orders))
print("order_items:", type(order_items))


분석할 데이터셋이 여러 개일 때는 딕셔너리로 묶어 두면 반복 점검을 하기 편합니다.


In [ ]:
datasets = {
    "customers": customers,
    "products": products,
    "orders": orders,
    "order_items": order_items,
}

list(datasets.keys())


## 4. 각 파일의 역할 이해하기

이번 과정에서 사용하는 데이터는 가상의 온라인 쇼핑몰 운영 데이터입니다.

| 파일 | 역할 | 먼저 확인할 것 |
| --- | --- | --- |
| `customers.csv` | 고객 정보 | 고객 수, 연령, 성별, 지역, 가입일 |
| `products.csv` | 상품 정보 | 상품 수, 카테고리, 가격 |
| `orders.csv` | 주문 정보 | 주문 수, 주문일, 결제수단, 주문상태 |
| `order_items.csv` | 주문 상세 정보 | 주문별 상품, 수량, 단가 |

처음에는 파일을 합치지 말고, 각 파일을 따로 살펴보는 것이 좋습니다.


![pandas DataFrame 구조 예시](../book/assets/images/ch03/ch03_dataframe_structure.svg)

DataFrame은 행(row)과 열(column)로 구성됩니다. `shape`, `head()`, `columns`, `info()` 같은 기본 도구를 사용해 구조를 확인합니다.


## 5. 데이터 크기 확인하기

`shape`는 데이터의 행과 열 개수를 알려 줍니다.

- 앞 숫자: 행 개수
- 뒤 숫자: 열 개수

예를 들어 `(150, 6)`은 150행 6열이라는 뜻입니다.


In [ ]:
print("customers:", customers.shape)
print("products:", products.shape)
print("orders:", orders.shape)
print("order_items:", order_items.shape)


In [ ]:
shape_summary = pd.DataFrame([
    {
        "dataset": name,
        "rows": df.shape[0],
        "columns": df.shape[1],
    }
    for name, df in datasets.items()
])

shape_summary


### 생각해 보기

- 가장 행이 많은 데이터셋은 무엇인가요?
- `order_items`가 `orders`보다 행이 많다면, 그 이유는 무엇일까요?
- 분석 보고서에 데이터 규모를 설명한다면 어떤 문장으로 쓸 수 있을까요?


## 6. 데이터 앞부분과 마지막 부분 보기

`head()`는 앞부분 5행을 보여 줍니다. 컬럼명이 예상과 맞는지, 값의 형태가 자연스러운지 빠르게 확인할 때 사용합니다.


In [ ]:
customers.head()


In [ ]:
products.head()


In [ ]:
orders.head()


In [ ]:
order_items.head()


앞부분만 보고 전체 데이터가 정상이라고 판단하기는 어렵습니다. `tail()`로 마지막 부분도 확인해 봅니다.


In [ ]:
customers.tail()


### 생각해 보기

`head()`와 `tail()`을 보면서 아래를 확인해 보세요.

- 날짜처럼 보이는 컬럼이 있나요?
- 숫자처럼 보이는 컬럼이 있나요?
- ID처럼 보이는 컬럼이 있나요?
- 사람이 직접 읽을 수 있는 이름이나 범주 값이 있나요?


## 7. 컬럼명 확인하기

컬럼명은 코드 작성에서 매우 중요합니다. 실제 컬럼명이 `customer_id`인데 LLM이나 사람이 `cust_id`라고 쓰면 코드는 실행되지 않습니다.


In [ ]:
for name, df in datasets.items():
    print(f"[{name}]")
    print(list(df.columns))
    print()


In [ ]:
column_summary = pd.DataFrame([
    {
        "dataset": name,
        "column_count": len(df.columns),
        "column_names": ", ".join(df.columns),
    }
    for name, df in datasets.items()
])

column_summary


### 생각해 보기

- 고객을 구분하는 컬럼은 무엇인가요?
- 주문을 구분하는 컬럼은 무엇인가요?
- 상품을 구분하는 컬럼은 무엇인가요?
- 여러 파일을 연결할 때 사용할 수 있을 것 같은 컬럼은 무엇인가요?


## 8. 데이터 타입 확인하기

`info()`는 컬럼별 데이터 타입과 비어 있지 않은 값의 개수를 보여 줍니다.

특히 날짜처럼 보이지만 `object`로 저장된 컬럼을 주의해서 봅니다. pandas에서 `object`는 보통 문자열 또는 여러 타입이 섞인 컬럼일 때 나타납니다.


In [ ]:
customers.info()


In [ ]:
for name, df in datasets.items():
    print(f"\n===== {name} =====")
    df.info()


In [ ]:
dtype_summary = pd.concat(
    [df.dtypes.rename(name) for name, df in datasets.items()],
    axis=1,
).fillna("")

dtype_summary


### 생각해 보기

- 숫자형 컬럼은 어떤 것들이 있나요?
- 문자형 컬럼은 어떤 것들이 있나요?
- 날짜처럼 보이지만 아직 문자열일 가능성이 있는 컬럼은 무엇인가요?


![데이터 구조 점검 흐름도](../book/assets/images/ch03/ch03_data_check_flow.svg)

데이터 구조 점검은 파일 확인, 로드, 크기 확인, 컬럼 확인, 타입 확인, 결측치 확인, 중복 확인, 키 관계 확인 순서로 진행하면 좋습니다.


## 9. 결측치 확인하기

결측치는 값이 비어 있는 상태입니다. 결측치가 있으면 평균, 비율, 그룹별 집계 결과가 달라질 수 있습니다.

`isna().sum()`은 컬럼별 결측치 개수를 계산합니다.


In [ ]:
customers.isna().sum()


In [ ]:
missing_summary = pd.concat(
    [df.isna().sum().rename(name) for name, df in datasets.items()],
    axis=1,
).fillna("").astype(str)

missing_summary


In [ ]:
missing_rate_summary = pd.concat(
    [(df.isna().mean() * 100).round(2).rename(name) for name, df in datasets.items()],
    axis=1,
).fillna("")

missing_rate_summary


### 결측치 해석 팁

결측치가 있다고 해서 무조건 삭제하는 것은 아닙니다.

| 처리 방법 | 설명 |
| --- | --- |
| 행 제외 | 결측치가 있는 행을 분석에서 제외합니다. |
| 대표값 대체 | 평균, 중앙값, 최빈값 등으로 채웁니다. |
| 별도 범주 처리 | `Unknown` 같은 범주로 표시합니다. |
| 컬럼 제외 | 분석 목적에 맞지 않는 컬럼은 사용하지 않습니다. |
| 원인 확인 | 수집 과정에서 문제가 있었는지 확인합니다. |


## 10. 중복 데이터 확인하기

중복은 같은 행이나 같은 ID가 반복되는 상태입니다.

단, 모든 중복이 오류는 아닙니다. 예를 들어 `order_items`에서는 한 주문에 여러 상품이 들어갈 수 있으므로 같은 `order_id`가 여러 번 나올 수 있습니다.


In [ ]:
duplicate_rows = pd.DataFrame([
    {
        "dataset": name,
        "duplicated_rows": df.duplicated().sum(),
    }
    for name, df in datasets.items()
])

duplicate_rows


In [ ]:
id_duplicate_checks = pd.DataFrame([
    {
        "check": "customers.customer_id",
        "duplicated_count": customers["customer_id"].duplicated().sum(),
        "interpretation": "0이어야 고객 ID가 유일합니다.",
    },
    {
        "check": "products.product_id",
        "duplicated_count": products["product_id"].duplicated().sum(),
        "interpretation": "0이어야 상품 ID가 유일합니다.",
    },
    {
        "check": "orders.order_id",
        "duplicated_count": orders["order_id"].duplicated().sum(),
        "interpretation": "0이어야 주문 ID가 유일합니다.",
    },
    {
        "check": "order_items.order_id",
        "duplicated_count": order_items["order_id"].duplicated().sum(),
        "interpretation": "한 주문에 여러 상품이 있으면 0보다 클 수 있습니다.",
    },
])

id_duplicate_checks


### 생각해 보기

- `customers.customer_id` 중복과 `order_items.order_id` 중복은 왜 의미가 다를까요?
- 중복 개수만 보고 삭제하면 위험한 이유는 무엇일까요?


## 11. 숫자형 컬럼 기본 통계 확인하기

`describe()`는 숫자형 컬럼의 개수, 평균, 표준편차, 최솟값, 사분위수, 최댓값을 보여 줍니다.

최솟값이나 최댓값이 지나치게 이상하면 데이터 오류나 이상치 가능성을 의심할 수 있습니다.


In [ ]:
customers.describe()


In [ ]:
products[["price"]].describe()


In [ ]:
order_items[["quantity", "unit_price"]].describe()


숫자가 문자열로 저장된 경우도 있습니다. 예를 들어 `"10,000"`처럼 쉼표가 포함된 문자열은 바로 계산하기 어렵습니다.


In [ ]:
price_text = pd.Series(["10,000", "25,500", "3000", "확인필요"])
price_number = pd.to_numeric(
    price_text.str.replace(",", "", regex=False),
    errors="coerce",
)

pd.DataFrame({
    "original": price_text,
    "converted": price_number,
})


`errors="coerce"`는 숫자로 바꿀 수 없는 값을 `NaN`으로 처리합니다. 변환 후에는 새로 생긴 결측치가 있는지도 확인해야 합니다.


## 12. 범주형 컬럼 고유값 확인하기

문자형 또는 범주형 컬럼은 고유값 개수와 빈도를 확인합니다. 예를 들어 지역, 성별, 카테고리, 주문 상태 같은 컬럼은 `value_counts()`로 분포를 볼 수 있습니다.


In [ ]:
customers["city"].value_counts().head(10)


In [ ]:
products["category"].value_counts()


In [ ]:
orders["order_status"].value_counts()


In [ ]:
categorical_summary = pd.DataFrame([
    {"dataset": "customers", "column": "city", "unique_count": customers["city"].nunique()},
    {"dataset": "customers", "column": "gender", "unique_count": customers["gender"].nunique()},
    {"dataset": "products", "column": "category", "unique_count": products["category"].nunique()},
    {"dataset": "orders", "column": "payment_method", "unique_count": orders["payment_method"].nunique()},
    {"dataset": "orders", "column": "order_status", "unique_count": orders["order_status"].nunique()},
])

categorical_summary


### 생각해 보기

- 특정 값에 데이터가 지나치게 몰려 있나요?
- 오타나 표기 차이처럼 보이는 값이 있나요?
- 나중에 그룹별 분석 기준으로 쓰기 좋은 컬럼은 무엇인가요?


## 13. 날짜 컬럼 확인하기

날짜 컬럼은 월별, 요일별, 기간별 분석에 자주 사용됩니다.

하지만 CSV에서 읽어온 날짜는 처음에는 문자열(`object`)일 수 있습니다. `pd.to_datetime()`으로 날짜 타입으로 바꿔야 날짜 계산을 안전하게 할 수 있습니다.


In [ ]:
orders["order_date"].head()


In [ ]:
print("변환 전 타입:", orders["order_date"].dtype)

orders["order_date"] = pd.to_datetime(orders["order_date"], errors="coerce")

print("변환 후 타입:", orders["order_date"].dtype)
print("날짜 변환 실패 건수:", orders["order_date"].isna().sum())
print("가장 빠른 주문일:", orders["order_date"].min())
print("가장 최근 주문일:", orders["order_date"].max())


In [ ]:
orders.assign(
    order_year=orders["order_date"].dt.year,
    order_month=orders["order_date"].dt.month,
    order_day_name=orders["order_date"].dt.day_name(),
).head()


### 생각해 보기

- 데이터는 어느 기간을 포함하고 있나요?
- 월별 매출 분석을 하기에 충분한 기간인가요?
- 날짜 변환 실패 건수가 0보다 크다면 무엇을 확인해야 할까요?


## 14. 여러 파일의 관계 확인하기

온라인 쇼핑몰 데이터는 고객, 상품, 주문, 주문 상세 데이터가 서로 연결되어야 분석할 수 있습니다.

| 연결 관계 | 의미 |
| --- | --- |
| `customers.customer_id` ↔ `orders.customer_id` | 어떤 고객이 주문했는지 연결합니다. |
| `orders.order_id` ↔ `order_items.order_id` | 주문과 주문 상세를 연결합니다. |
| `products.product_id` ↔ `order_items.product_id` | 주문 상세와 상품 정보를 연결합니다. |


![4개 CSV 파일 간 키 관계도](../book/assets/images/ch03/ch03_csv_key_relationships.svg)


In [ ]:
invalid_customers = orders[~orders["customer_id"].isin(customers["customer_id"])]
invalid_orders = order_items[~order_items["order_id"].isin(orders["order_id"])]
invalid_products = order_items[~order_items["product_id"].isin(products["product_id"])]

relationship_check = pd.DataFrame([
    {
        "relationship": "orders.customer_id -> customers.customer_id",
        "invalid_rows": len(invalid_customers),
    },
    {
        "relationship": "order_items.order_id -> orders.order_id",
        "invalid_rows": len(invalid_orders),
    },
    {
        "relationship": "order_items.product_id -> products.product_id",
        "invalid_rows": len(invalid_products),
    },
])

relationship_check


`invalid_rows`가 모두 0이면 샘플 데이터에서는 기본적인 연결 관계가 유지되고 있다고 볼 수 있습니다. 0보다 큰 값이 있다면 어느 파일에서 기준 ID가 빠져 있는지 먼저 확인해야 합니다.


## 15. 간단한 병합으로 관계 확인하기

키 관계가 맞는지 확인한 뒤에는 데이터를 병합해 볼 수 있습니다. 아래 코드는 주문 상세(`order_items`)에 상품 정보(`products`)를 붙이고, 각 행의 금액을 계산합니다.


In [ ]:
order_items_with_products = order_items.merge(
    products,
    on="product_id",
    how="left",
)

order_items_with_products["line_amount"] = (
    order_items_with_products["quantity"] * order_items_with_products["unit_price"]
)

order_items_with_products.head()


In [ ]:
category_sales = (
    order_items_with_products
    .groupby("category", as_index=False)["line_amount"]
    .sum()
    .sort_values("line_amount", ascending=False)
)

category_sales


이 결과는 본격적인 EDA가 아니라, 데이터 관계가 실제로 연결되는지 확인하는 작은 점검입니다. 분석 결론을 내리기 전에 결측치, 주문 상태, 취소 주문 처리 기준 등을 더 확인해야 합니다.


## 16. 반복 점검을 함수로 정리하기

여러 데이터셋에 같은 점검을 반복할 때는 함수로 정리하면 편합니다.


In [ ]:
def check_data_overview(name: str, df: pd.DataFrame) -> None:
    print(f"===== {name} =====")
    print("shape:", df.shape)
    print("\ncolumns:")
    print(list(df.columns))
    print("\ndtypes:")
    print(df.dtypes)
    print("\nmissing values:")
    print(df.isna().sum())
    print("\nduplicated rows:", df.duplicated().sum())


check_data_overview("customers", customers)


In [ ]:
for name, df in datasets.items():
    check_data_overview(name, df)
    print()


![Jupyter Notebook 데이터 구조 점검 결과 화면 예시](../book/assets/images/ch03/ch03_jupyter_data_overview_result.svg)


## 17. LLM에게 데이터 구조를 설명시키는 법

LLM에게 원본 데이터를 그대로 붙여 넣는 것은 피하는 것이 좋습니다. 대신 아래처럼 구조 요약만 전달합니다.

- 파일명
- 컬럼명
- 행과 열 개수
- 데이터 타입
- 결측치 개수
- 중복 여부
- 파일 간 키 관계


In [ ]:
llm_dataset_summary = shape_summary.merge(column_summary, on="dataset")
llm_dataset_summary


In [ ]:
for _, row in llm_dataset_summary.iterrows():
    print(f"- {row['dataset']}: {row['rows']}행 {row['columns']}열")
    print(f"  컬럼: {row['column_names']}")


### 데이터 구조 설명 요청 예시

아래 프롬프트는 LLM에게 붙여 넣을 수 있는 예시입니다. 실제 데이터 전체가 아니라 구조 정보만 포함합니다.


In [ ]:
prompt = f"""
온라인 쇼핑몰 데이터 분석을 시작하기 전에 다음 CSV 파일들의 구조를 이해하려고 합니다.

데이터셋 요약:
{llm_dataset_summary[['dataset', 'rows', 'columns', 'column_names']].to_string(index=False)}

파일 간 관계:
- customers.customer_id -> orders.customer_id
- orders.order_id -> order_items.order_id
- products.product_id -> order_items.product_id

요청:
1. 각 파일이 어떤 역할을 하는지 설명해 주세요.
2. 분석 전에 확인해야 할 항목을 체크리스트로 정리해 주세요.
3. 실제 데이터 확인 없이 단정한 내용과 추가 확인이 필요한 내용을 구분해 주세요.
"""

print(prompt)


## 18. LLM 답변 검증 연습

LLM이 다음과 같이 답했다고 가정해 봅니다.

> 고객 데이터에 age 컬럼이 있으므로 연령대별 매출 분석을 바로 수행하면 됩니다.

이 답변은 그럴듯하지만 충분히 안전하지 않습니다. 아래 내용을 직접 확인해야 합니다.

- `age` 컬럼이 실제로 존재하는가?
- `age` 컬럼에 결측치나 이상치가 있는가?
- 고객 데이터와 주문 데이터가 `customer_id`로 연결되는가?
- 매출을 계산하려면 주문 상세와 상품 또는 단가 정보가 필요한가?
- 취소 주문을 포함할지 제외할지 기준이 있는가?


In [ ]:
validation_check = pd.DataFrame([
    {
        "question": "customers에 age 컬럼이 있는가?",
        "result": "age" in customers.columns,
    },
    {
        "question": "age 결측치 개수는?",
        "result": customers["age"].isna().sum() if "age" in customers.columns else "컬럼 없음",
    },
    {
        "question": "orders.customer_id가 customers.customer_id와 연결되는가?",
        "result": len(invalid_customers) == 0,
    },
    {
        "question": "매출 계산에 필요한 quantity와 unit_price가 있는가?",
        "result": {"quantity", "unit_price"}.issubset(order_items.columns),
    },
])

validation_check


## 19. 이번 장 점검 체크리스트

| 점검 항목 | 확인 |
| --- | --- |
| 필요한 CSV 파일이 모두 존재하는가? | □ |
| 각 데이터셋의 행과 열 개수를 확인했는가? | □ |
| 컬럼명이 예상과 일치하는가? | □ |
| 날짜 컬럼의 데이터 타입을 확인했는가? | □ |
| 숫자 컬럼이 실제 숫자형으로 저장되어 있는가? | □ |
| 결측치가 있는 컬럼을 확인했는가? | □ |
| 중복 데이터가 있는지 확인했는가? | □ |
| 주요 ID 컬럼의 중복 여부를 확인했는가? | □ |
| 여러 파일을 연결할 키 컬럼을 확인했는가? | □ |
| 파일 간 키 관계가 실제로 연결 가능한지 확인했는가? | □ |
| LLM에 원본 데이터 대신 구조 요약만 입력했는가? | □ |
| LLM이 제안한 설명을 실제 데이터와 비교해 검증했는가? | □ |


## 20. 실습 과제

아래 과제를 직접 해결해 보세요.

1. 4개 CSV 파일의 행과 열 개수를 하나의 표로 정리하세요.
2. 각 파일의 결측치 개수와 결측치 비율을 확인하세요.
3. `orders.order_date`를 날짜 타입으로 변환하고 데이터 기간을 확인하세요.
4. `order_items`와 `products`를 병합해 카테고리별 주문 금액을 계산하세요.
5. LLM에게 데이터 구조 요약을 전달하는 프롬프트를 직접 작성하세요.
6. LLM이 만든 분석 아이디어가 실제 컬럼과 키 관계에 맞는지 검증하세요.


In [ ]:
# 과제 풀이 공간입니다.
# 필요한 코드를 직접 작성해 보세요.


## 마무리

이번 장의 핵심은 “데이터를 불러왔다”에서 끝내지 않는 것입니다.

데이터 분석을 시작하기 전에 파일, 행과 열, 컬럼명, 데이터 타입, 결측치, 중복, 키 관계를 확인해야 이후 분석이 흔들리지 않습니다. 다음 장에서는 이 구조를 바탕으로 pandas의 선택, 필터링, 정렬, 집계 기초를 다룹니다.
